In [ ]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.statistics import multivariate_logrank_test, logrank_test
from lifelines.utils import restricted_mean_survival_time
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import plotnine as p9
import copy

from src.clustering.clinical_relevance_clusters import get_p_rmst
from src.clustering.settings import best_pipeline, clinical_endpoints, timepoints, discrete_clinical_variables, final_cluster_name, rename_cols_dict

plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
clinical_endpoints_description = [val["description"] for val in clinical_endpoints.values()]
discrete_clinical_variables = [rename_cols_dict[val] for val in discrete_clinical_variables]

In [ ]:
embedding_clusters = os.path.join("results_v2_permodality", "clustering", "embedding_clusters.csv")
embedding_clusters = pd.read_csv(embedding_clusters, index_col=0)
embedding_clusters = embedding_clusters[best_pipeline["name"]].rename(final_cluster_name)
embedding_clusters = embedding_clusters.replace({0: "Low-risk", 1: "High-risk"})
print(f"{embedding_clusters.shape=}")
embedding_clusters.head()

In [ ]:
clinical_data = os.path.join("outcomes", "all_fields_processed.csv")
clinical_data = pd.read_csv(clinical_data)
clinical_data = clinical_data.rename(columns=rename_cols_dict)
print(f"{clinical_data.shape=}")
clinical_data.head()

In [ ]:
clinical_data_with_clusters = clinical_data.merge(embedding_clusters.to_frame(final_cluster_name).reset_index(names="patient_id"), on="patient_id")

In [ ]:
df_survival = []
temporal_results = []
logrank_test_list = []
for row_idx, var_comp in enumerate([final_cluster_name] + discrete_clinical_variables):
    for clinical_endpoint_idx, clinical_endpoint in enumerate(clinical_endpoints):
        t = clinical_endpoints[clinical_endpoint]["t"]
        e = clinical_endpoints[clinical_endpoint]["e"]
        description = clinical_endpoints[clinical_endpoint]["description"]
        clinical_data_endpoint = clinical_data_with_clusters[
            clinical_data_with_clusters[t].notna() &
            clinical_data_with_clusters[e].notna() &
            clinical_data_with_clusters[var_comp].notna()
            ]
        clinical_data_endpoint = clinical_data_endpoint.sort_values(by=t, ascending=False)
        clinical_data_endpoint = clinical_data_endpoint.drop_duplicates(subset='patient_id', keep='first')
        clinical_data_endpoint = clinical_data_endpoint.set_index("patient_id")

        clusters_for_var = clinical_data_endpoint[var_comp]
        if clusters_for_var.nunique() > 2:
            continue

        if row_idx == 0:
            logrank_test = multivariate_logrank_test(
                event_durations=clinical_data_endpoint[t],
                event_observed=clinical_data_endpoint[e],
                groups=clusters_for_var
            )
            
            logrank_test_list.append({"endpoint": description, "p": logrank_test.p_value})

        kmfs = []
        maxtime_list = []
        for label_id, label in enumerate(sorted(clusters_for_var.unique())):
            cl = (clusters_for_var == label)
            nn = len(clinical_data_endpoint[t][cl])
            ne = int(clinical_data_endpoint[e][cl].sum())
            kmf_cl = KaplanMeierFitter().fit(
                durations=clinical_data_endpoint[t][cl],
                event_observed=clinical_data_endpoint[e][cl],
                label=f"{label}"
            )
            kmfs.append(kmf_cl)
            maxtime_list.append(int(kmf_cl.timeline[-1].round()))
            if row_idx == 0:
                kmf_cl_copy = copy.deepcopy(kmf_cl)
                km_df = kmf_cl_copy.survival_function_
                km_df.columns = ["Survival Probability"]
                cols = kmf_cl_copy.confidence_interval_.columns
                km_df['lower_ci'] = kmf_cl_copy.confidence_interval_[cols[0]]
                km_df['upper_ci'] = kmf_cl_copy.confidence_interval_[cols[1]]
                km_df[final_cluster_name] = label
                km_df["endpoint"] = description
                km_df = km_df.reset_index(names="Time (months)")
                df_survival.append(km_df)

        if clinical_endpoint_idx == 0:
            id_cl = np.argmax(maxtime_list)
            id_others = np.argmin(maxtime_list)
        kmf_cl = kmfs[id_cl]
        kmf_others = kmfs[id_others]

        rmst_list = []
        pval_list = []
        cox_pvals_list = []
        cox_coeff_list = []
        final_timepoints_list = []
        for timepoint in timepoints:
            if timepoint > min(maxtime_list):
                break
            rmst = restricted_mean_survival_time(kmf_cl, t=timepoint) - restricted_mean_survival_time(kmf_others, t=timepoint)
            pval = get_p_rmst(diff=rmst, kmf1=kmf_cl, kmf0=kmf_others, tau=timepoint)

            mask = clinical_data_endpoint[t] <= timepoint
            df_cox = clinical_data_endpoint[mask]
            df_cox["var"] = clusters_for_var[mask].map({kmf_cl._label: 1, kmf_others._label: 0}) if clusters_for_var.dtype == object else clusters_for_var[mask]
            cph = CoxPHFitter()
            cph.fit(df_cox[["var", t, e]], duration_col=t, event_col=e)
            cox_pvals = cph.summary["p"][0]
            cox_coeff = cph.summary["exp(coef)"][0]
            
            rmst_list.append(rmst)
            pval_list.append(pval)
            cox_pvals_list.append(cox_pvals)
            cox_coeff_list.append(cox_coeff)
            final_timepoints_list.append(timepoint)

        pval_list = multipletests(pval_list, method='fdr_bh')[1]
        cox_pvals_list = multipletests(cox_pvals_list, method='fdr_bh')[1]
        sign = np.where(pval_list <= 0.05, "Yes", "No")
        sign_cox = np.where(cox_pvals_list <= 0.05, "Yes", "No")
        logpval_list = -np.log10(pval_list)
        logpvalcox_list = -np.log10(cox_pvals_list)
        temporal_results_dict = {
            "ΔRMST": rmst_list,
            "adjp": pval_list,
            "adjp<0.05": sign,
            "-log(adjp)": logpval_list,
            "adjp_cox": cox_pvals_list,
            "adjp<0.05_cox": sign_cox,
            "-log(adjp_cox)": logpvalcox_list,
            "Hazard ratio": cox_coeff_list,
            "Time (months)": final_timepoints_list,
        }
        temporal_results_dict = pd.DataFrame(temporal_results_dict)
        temporal_results_dict["endpoint"] = description
        temporal_results_dict["Variable"] = f"{var_comp}: {kmf_cl._label} vs  {kmf_others._label}"
        temporal_results.append(temporal_results_dict)
temporal_results = pd.concat(temporal_results)
temporal_results["endpoint"] = pd.Categorical(temporal_results["endpoint"], categories=clinical_endpoints_description, ordered=True)

In [ ]:
x,y = "Time (months)", "Survival Probability"
facet_cols = "endpoint"
fill_by = final_cluster_name
plot_data = pd.concat(df_survival)
plot_data["endpoint"] = pd.Categorical(plot_data["endpoint"], categories=clinical_endpoints_description, ordered=True)

logrank_test_df = pd.DataFrame(logrank_test_list)
logrank_test_df["adjp"] = multipletests(logrank_test_df["p"], method='fdr_bh')[1]
logrank_test_df["sig"] = logrank_test_df["adjp"].lt(0.05)
logrank_test_df["adjp"] = "adjp = " + logrank_test_df["adjp"].map('{:.3f}'.format)
logrank_test_df["x"] = 50
logrank_test_df["y"] = 0.55
logrank_test_df["endpoint"] = pd.Categorical(logrank_test_df["endpoint"], categories=clinical_endpoints_description, ordered=True)

p_km = (
    p9.ggplot(plot_data, p9.aes(x=x, y=y, group=fill_by, fill=fill_by, color=fill_by))
    + p9.geom_step(size=1.2)
    + p9.geom_ribbon(p9.aes(ymin='lower_ci', ymax='upper_ci'), alpha=0.25)
    + p9.facet_wrap(facet_cols, nrow=1)
    + p9.geom_text(p9.aes(x="x", y="y", label="adjp"), data=logrank_test_df[logrank_test_df["sig"]], color="red", inherit_aes=False)
    + p9.geom_text(p9.aes(x="x", y="y", label="adjp"), data=logrank_test_df[~logrank_test_df["sig"]], color="black", inherit_aes=False)
    + p9.ylim(0.5, 1)
    + p9.theme_classic()
    + p9.theme(strip_background=p9.element_blank(), legend_position="top", legend_box_spacing=0.01)
    + p9.labs(tag="a", title="Kaplan–Meier survival curves by cluster")
)
p_km

In [ ]:
x,y = "Time (months)", "ΔRMST"
facet_cols = "endpoint"
fill_by = "Variable"
size = "-log(adjp)"
p_rmst = (
    p9.ggplot(temporal_results, p9.aes(x=x, y=y, fill=fill_by, color=fill_by))
    + p9.geom_line(size=1)
    + p9.geom_point(p9.aes(size=size))
    + p9.facet_wrap(facet_cols, nrow=1)
    + p9.theme_classic()
    + p9.theme(legend_position="top", legend_box='vertical', legend_box_just='center', strip_background=p9.element_blank(),
               axis_title_x=p9.element_blank(), axis_text_x=p9.element_blank(), axis_ticks_x=p9.element_blank(), legend_box_spacing=0.01)
    + p9.scale_fill_brewer(type="qualitative", palette="Set1", direction=1)
    + p9.scale_color_brewer(type="qualitative", palette="Set1", direction=1)
    + p9.scale_size_continuous(range=(0, 5), breaks=[2, 3, 4])
    + p9.guides(fill=p9.guide_legend(nrow=2))
    + p9.labs(tag="b", title="Clinical relevance over time", size="-log\u2081\u2080(adjp)")
)
p_rmst

In [ ]:
x,y = "Time (months)", "Hazard ratio"
facet_cols = "endpoint"
fill_by = "Variable"
size = "-log(adjp_cox)"
p_cox = (
    p9.ggplot(temporal_results, p9.aes(x=x, y=y, fill=fill_by, color=fill_by))
    + p9.geom_line(size=1)
    + p9.geom_point(p9.aes(size=size))
    + p9.facet_wrap(facet_cols, nrow=1)
    + p9.theme_classic()
    + p9.theme(legend_position="none", strip_background=p9.element_blank(), strip_text=p9.element_blank())
    + p9.scale_fill_brewer(type="qualitative", palette="Set1", direction=1)
    + p9.scale_color_brewer(type="qualitative", palette="Set1", direction=1)
    + p9.scale_size_continuous(range=(0, 5), breaks=[2, 3, 4])
)
p_cox

In [ ]:
x,y = "Status", "count"
facet_cols = "Column"
fill_by = final_cluster_name
color_by = final_cluster_name

count_groups_df = pd.melt(
    clinical_data_with_clusters, 
    id_vars=final_cluster_name, 
    value_vars=discrete_clinical_variables, 
    var_name='Column', 
    value_name='Status'
)
count_groups_df = count_groups_df.value_counts().reset_index()

pvals_list = []
for clin_var in count_groups_df["Column"].unique():
    mask = clinical_data_with_clusters[clin_var].notnull()
    contingency_table = pd.crosstab(
        index=clinical_data_with_clusters.loc[mask, final_cluster_name],
        columns=clinical_data_with_clusters.loc[mask, clin_var]
    )
    p = chi2_contingency(contingency_table)[1]
    pvals_dict = {
        "Column": clin_var,
        "p": p,
        "x": clinical_data_with_clusters.loc[mask, clin_var].nunique()/2+0.5,
        "y": 592.5,
    }
    pvals_list.append(pvals_dict)
    
pvals_dict = pd.DataFrame(pvals_list)
pvals_dict["adjp"] = multipletests(pvals_dict["p"], method='fdr_bh')[1]
pvals_dict["sig"] = pvals_dict["adjp"].lt(0.05)
pvals_dict["adjp"] = "adjp = " + pvals_dict["adjp"].map("{:.2f}".format)

count_groups_df["Column"] = count_groups_df["Column"].replace(rename_cols_dict)
pvals_dict["Column"] = pvals_dict["Column"].replace(rename_cols_dict)

count_groups_df["total_in_status"] = count_groups_df.groupby(["Column", "Status"])["count"].transform("sum")
count_groups_df["percentage"] = (count_groups_df["count"] / count_groups_df["total_in_status"]) * 100
target_cluster = count_groups_df[fill_by].unique()[0]
count_groups_df["percentage_label"] = count_groups_df.apply(lambda row: f"{round(row['percentage'])}%" if row[fill_by] == target_cluster else "", axis=1)

p_clin = (
    p9.ggplot(count_groups_df, p9.aes(x=x, y=y, fill=fill_by)) 
    + p9.geom_col(position="stack")
    + p9.facet_wrap(facet_cols, nrow=1, scales="free_x")
    + p9.geom_text(p9.aes(x="x", y="y", label="adjp"), data=pvals_dict[pvals_dict["sig"]], color="red", inherit_aes=False)
    + p9.geom_text(p9.aes(x="x", y="y", label="adjp"), data=pvals_dict[~pvals_dict["sig"]], color="black", inherit_aes=False)
    + p9.geom_text(p9.aes(label="percentage_label"), position=p9.position_stack(vjust=0.5), color="white")
    + p9.theme_classic()
    + p9.theme(legend_position="top", strip_background=p9.element_blank(), axis_title_x=p9.element_blank(), legend_box_spacing=0.02)
    + p9.labs(tag="c", title="Association between clusters and clinical variables", y="Patients")
)
p_clin

In [ ]:
composition = p_km / p_rmst / p_cox / p_clin + p9.theme(figure_size=(11, 13))
composition

In [ ]:
composition.save(os.path.join("figures", "Figure4.svg"))
composition.save(os.path.join("figures", "Figure4.pdf"))